In [1]:
import sys
import scipy.stats as stats
import re
import matplotlib.colors as mcolors

import builtins
builtins.NAMING = "new"

sys.path.insert(0, "../../")
from RealTrace_paper_analysis.plotting_src.header_settings import *

from RealTrace_paper_analysis.plotting_src.read_ggp_run import * # you might need to change these two line, such that you can use read ggp_run
from RealTrace_paper_analysis.plotting_src.cells_utils import * # you might need to change these two line, such that you can use read ggp_run
from RealTrace_paper_analysis.plotting_src.read_utils import * # you might need to change these two line, such that you can use read ggp_run

%config InlineBackend.figure_format ='retina'
%reload_ext autoreload
%autoreload 2



/var/folders/_h/4605456d743bfjqsq6r1jcdm0000gp/T/ipykernel_4866/3060108594.py:2: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 1.21.0)
  import scipy.stats as stats


header_setting.py: using new naming


In [2]:
def exent_parameter_settings(ps):
    # add dummy "error" if not optimized
    for key in ps.keys():
        if len(ps[key]) == 2:
            ps[key].append(0)

    for rate in ["lambda", "q"]:
        std = np.sqrt(ps["var_"+rate][1]/(2.*ps["gamma_"+rate][1]))
        std_err2 = (1/2. * 1/np.sqrt( ps["var_"+rate][1]*ps["gamma_"+rate][1]) * ps["var_"+rate][2])**2 + \
                        (1/2. * np.sqrt(ps["var_"+rate][1]/2.) * ps["gamma_"+rate][1]**(-3/2.)* ps["gamma_"+rate][2])**2
        ps["std_"+rate] = ["derived", std, np.sqrt(std_err2)]

    for rate in ["lambda", "q"]:
        sigma = np.sqrt(ps["var_"+rate][1])
        ps["sigma_"+rate] = ["derived", sigma, None]
        
    for xg in ["dx", "dg"]:
        sigma = np.sqrt(ps["var_"+xg][1])
        ps["sigma_"+xg] = ["derived", sigma, None]
        
    for xg in ["x", "g"]:
        sigma = np.sqrt(ps["var_"+xg][1])
        sigma_err = 1/np.sqrt(ps["var_"+xg][1]) * ps["var_"+xg][2]
        ps["sigma_"+xg] = ["derived", sigma, sigma_err]
        
    return ps

In [3]:

prediction_files_ecoli = sort_by_condition(get_suffix_files("../../data_realtrace_paper/ecoli_data/", suffix='prediction.csv')) 
prediction_files_celegans = get_suffix_files("../../data_realtrace_paper/celegans_data/", suffix='prediction.csv')
prediction_files_esc = get_suffix_files("../../data_realtrace_paper/esc_data/", suffix='prediction.csv')
# prediction_files_Kennard = ['../../ecoli_other_labs/Kennard2016/RealTrace_out_filter_r2/Data_Glc_P5ori_processed_filter_r2_f8_b0123457_prediction.csv']
# prediction_files_Kar = get_suffix_files("../../ecoli_other_labs/Kar2021/RealTrace_out/", suffix='prediction.csv') 


# plot_dir = mk_missing_dir(os.path.join( *prediction_files[0].split('/')[:-1], "plots" ))
prediction_files = np.concatenate([prediction_files_ecoli, 
                                   prediction_files_celegans, 
                                   prediction_files_esc,])
                                   # prediction_files_Kennard,
                                   # prediction_files_Kar])
len(prediction_files)

14

### Supplement parameter estimates

In [4]:
def format_number_for_latex(n, magnitude_lim=[-3, 3], sign_dig=3):
    formatting_str_for_e ="{:." + str(sign_dig-1) +"e}"
    e_str = formatting_str_for_e.format(n)
    oom = int(e_str.split("e")[-1]) #order of magnitude
    
    if oom < magnitude_lim[0] or oom > magnitude_lim[1]:
        q = e_str.split("e")[0]
        return "$"+q + r" \times 10^{" + str(oom) +"}$"

    formatting_no = np.max([sign_dig-1-oom, 0])
    formatting_str_for_f = "${:."+str(formatting_no)+"f}$"
    return formatting_str_for_f.format(n)

    
def write_latex_table(prediction_files, outfile):
    with open(outfile, "w", encoding="utf-8") as f:
        f.write("%table generated by supp_parameter_table.ipynb\n")
        f.write(r"\textbf{Data set} & " 
                +r"$\bar{\lambda}$(1/min) & $\gamma_{\lambda}$(1/min)  & $\sigma^2_{\lambda}$(1/min$^2$) & "
                +r"$\bar{q}$(mol./min) & $\gamma_q$(1/min)  & $\sigma^2_{q}$(mol./min$^2$) & " 
                +r"$\beta$(1/min) & "
                +r"$\sigma^2_{x}$ & $\sigma^2_{g}$ & "
                +r"$\sigma^2_{dx}$ & $\sigma^2_{dg}$ \\ \hline\hline ")
        f.write("\n")  
                
        for i, prediction_file in enumerate(prediction_files):
            inferred = read_header(prediction_file)
            inferred = exent_parameter_settings(inferred)
            parameters_to_write = [param for param in inferred.keys() if inferred[param][0] in ["free", "fixed"]]
            if prediction_file in prediction_files_ecoli:
                cond = label_by_condition[prediction_file.split("/")[-1].split("_")[0]]
                f.write(r"\textit{E.coli} "+cond +", ")
                prom = prediction_file.split("/")[-1].split("_")[1]
                f.write(r"\textit{" +prom +"} & ")
            if prediction_file in prediction_files_celegans:
                f.write(r"\hline \textit{C.elegans} &")

            if prediction_file in prediction_files_esc:
                f.write(r" \hline Mouse e.s.c &")

            # if prediction_file in prediction_files_Kennard:
            #     f.write(r" \hline Kennard et al. 2016 &")
            
            # if prediction_file in prediction_files_Kar:
            #     f.write(r" \hline Kar et al. 2021 &")
                
            for i, param in enumerate(parameters_to_write):
                if inferred[param][0] not in ["free", "bound", "fixed"]:
                    continue
                f.write(format_number_for_latex(inferred[param][1])) 
                if i<len(parameters_to_write)-1:
                    f.write(" & ")
                else:
                    f.write(" \\\\ \n")
        f.write(r"\hline")
                    

write_latex_table(prediction_files, outfile="../../figures_for_method_paper/supp/parameter_table.txt")
